<a href="https://colab.research.google.com/github/AlaaTaima/plant-model/blob/alaa/plantmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#1- ربط Drive
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/plant_project/"
os.makedirs(BASE, exist_ok=True)

Mounted at /content/drive


In [2]:
#الخطوة 1: الصور التلقائية (شغّلها في Colab بعد ربط Drive)
import random, shutil, glob, os
BASE = "/content/drive/MyDrive/plant_project/"
out = BASE + "external/auto_unknown/"
os.makedirs(out, exist_ok=True)

!wget -q https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz -O /content/flowers.tgz
!tar -xzf /content/flowers.tgz -C /content
!wget -q https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip -O /content/catsdogs.zip
!unzip -q /content/catsdogs.zip -d /content

pool = (glob.glob("/content/flower_photos/*/*.jpg") +
        glob.glob("/content/cats_and_dogs_filtered/train/*/*.jpg"))
random.seed(0)
for i, p in enumerate(random.sample(pool, 150)):
    shutil.copy(p, out + f"auto_{i:03d}.jpg")
print("عدد الصور:", len(os.listdir(out)))      # 150

[/content/catsdogs.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/catsdogs.zip or
        /content/catsdogs.zip.zip, and cannot find /content/catsdogs.zip.ZIP, period.
عدد الصور: 150


In [3]:
#1- تنزيل الصور

!pip install -q bing-image-downloader

import os, shutil
from bing_image_downloader import downloader

BASE = "/content/drive/MyDrive/plant_project/"      # شغّل خلية ربط Drive قبلها
DL = "/content/dl"
shutil.rmtree(DL, ignore_errors=True)

groups = {
    # نفس نباتات الداتا لكن بصور حقلية (أهم مجموعة)
    "field_same_plants": [
        "tomato plant leaves in garden", "bell pepper plant leaves garden",
        "corn leaves in field", "apple tree leaves orchard",
        "grape vine leaves vineyard", "potato plant leaves garden",
        "strawberry plant leaves garden", "soybean plant leaves field",
        "squash plant leaves garden", "peach tree leaves", "cherry tree leaves",
        "orange tree leaves citrus", "blueberry bush leaves", "raspberry plant leaves",
        "hand holding diseased tomato leaf", "tomato leaf disease outdoor",
    ],
    # نباتات ليست في الداتا
    "other_plants": [
        "rose leaves garden", "basil plant", "lettuce plant garden", "banana leaf",
        "oak tree leaves", "cactus plant", "houseplant leaves", "mint plant leaves",
    ],
    # أشياء عشوائية
    "objects": [
        "kitchen table", "person hand close up", "soil ground", "street road",
        "wooden floor", "city building", "bicycle", "coffee cup",
    ],
}

for group, queries in groups.items():
    for q in queries:
        try:
            downloader.download(q, limit=10,
                                output_dir=f"{DL}/{group}",
                                adult_filter_off=False,
                                force_replace=False,
                                timeout=20,
                                filter="photo",
                                verbose=False)
        except Exception as e:
            print("تعذّر:", q, "|", type(e).__name__)
print("انتهى التنزيل")

[%] Downloading Images to /content/dl/field_same_plants/tomato plant leaves in garden
[!] Issue getting: https://preview.redd.it/whats-up-with-these-leaves-on-my-tomato-plant-v0-emq01oe2542b1.jpg?width=1080&crop=smart&auto=webp&s=72565934b0c60bd8a0688cecb8694be8061da10b
[!] Error:: HTTP Error 403: Blocked
[!] Issue getting: https://preview.redd.it/why-are-the-new-leaves-on-my-tomato-plant-yellow-v0-zg0ognepws7d1.jpg?width=1840&format=pjpg&auto=webp&s=8eb5ff8019834daf13b6a147fa5911790859cbab
[!] Error:: HTTP Error 403: Blocked


[%] Done. Downloaded 10 images.
[%] Downloading Images to /content/dl/field_same_plants/bell pepper plant leaves garden


[%] Done. Downloaded 10 images.
[%] Downloading Images to /content/dl/field_same_plants/corn leaves in field
[!] Issue getting: https://thumbs.dreamstime.com/z/corn-leaves-grows-field-292027247.jpg
[!] Error:: HTTP Error 500: Internal Server Error


[%] Done. Downloaded 10 images.
[%] Downloading Images to /content/dl/field_same_plants/apple 

In [4]:
import os
print("المتغيرات موجودة:", all(v in globals() for v in ("groups", "DL", "BASE")))

for g in ("field_same_plants", "other_plants", "objects"):
    base = f"/content/dl/{g}"
    if not os.path.isdir(base):
        print(f"{g:20s} غير موجود"); continue
    total = sum(len(os.listdir(os.path.join(base, q))) for q in os.listdir(base))
    print(f"{g:20s} استعلامات: {len(os.listdir(base)):2d} | صور: {total}")

المتغيرات موجودة: True
field_same_plants    استعلامات: 16 | صور: 160
other_plants         استعلامات:  8 | صور: 80
objects              استعلامات:  8 | صور: 80


In [5]:
#2- التنظيف والنسخ إلى Drive


import glob, hashlib
from PIL import Image

def clean_and_copy(src_root, dst_root, min_side=128):
    os.makedirs(dst_root, exist_ok=True)
    seen, kept, dropped = set(), 0, 0
    for p in sorted(glob.glob(src_root + "/**/*", recursive=True)):
        if os.path.isdir(p):
            continue
        try:
            with open(p, "rb") as f:
                h = hashlib.md5(f.read()).hexdigest()
            if h in seen:
                dropped += 1; continue
            im = Image.open(p).convert("RGB")
            if min(im.size) < min_side:
                dropped += 1; continue
            seen.add(h)
            q = os.path.basename(os.path.dirname(p)).replace(" ", "_")[:30]
            im.save(os.path.join(dst_root, f"{q}_{kept:04d}.jpg"), quality=95)
            kept += 1
        except Exception:
            dropped += 1
    return kept, dropped

for group in groups:
    kept, dropped = clean_and_copy(f"{DL}/{group}", f"{BASE}external/{group}")
    print(f"{group:20s} محفوظة: {kept:4d} | محذوفة: {dropped}")

field_same_plants    محفوظة:  160 | محذوفة: 0


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


other_plants         محفوظة:   80 | محذوفة: 0
objects              محفوظة:   80 | محذوفة: 0


In [7]:
#3- معاينة الصور لتحذف غير المناسب

import glob, os

def delete_ids(group, ids):
    for i in ids:
        for p in glob.glob(f"{BASE}external/{group}/*_{i:04d}.jpg"):
            os.remove(p); print("حُذف:", os.path.basename(p))

delete_ids("field_same_plants", [18])

حُذف: bell_pepper_plant_leaves_garde_0018.jpg


In [ ]:
#2- تحميل الداتا
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_a19db74358b723fba726eb9370551b93"

!pip install -q "kaggle>=1.8.0"
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content
!unzip -q /content/plantvillage-dataset.zip "plantvillage dataset/color/*" -d /content

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
 69% 1.40G/2.04G [00:09<00:10, 64.3MB/s]

In [ ]:
#3- التحقق من التحميل
LOCAL = "/content/plantvillage dataset/color"
classes = sorted(os.listdir(LOCAL))
total = sum(len(os.listdir(os.path.join(LOCAL, c))) for c in classes)
print(len(classes), total)      # المتوقع: 38 54305

In [ ]:
# 4. فحص الصور وبناء الجدول
#تعرّف الدالة info(item) التي تفحص صورة واحدة: تقرأ الملف مرة واحدة وتحسب منه بصمة
#MD5، ثم تتحقق من سلامته بـ verify()، وتستخرج الأبعاد ونمط الألوان. إذا فشل شيء تضع corrupted=True

import io, hashlib, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

def info(item):
    p, c = item
    row = {"path": p, "label": c, "fname": os.path.basename(p),
           "bytes": os.path.getsize(p), "width": None, "height": None,
           "mode": None, "corrupted": False, "hash": None}
    try:
        with open(p, "rb") as f:
            data = f.read()
        row["hash"] = hashlib.md5(data).hexdigest()
        Image.open(io.BytesIO(data)).verify()
        im = Image.open(io.BytesIO(data))
        row["width"], row["height"] = im.size
        row["mode"] = im.mode
    except Exception:
        row["corrupted"] = True
    return row

items = [(os.path.join(LOCAL, c, f), c)
         for c in classes for f in os.listdir(os.path.join(LOCAL, c))]

with ThreadPoolExecutor(8) as ex:
    rows = list(tqdm(ex.map(info, items), total=len(items)))

df = pd.DataFrame(rows)
print(df.shape)     # المتوقع (54305, 9)

In [ ]:

#5- التنظيف
df = df.sort_values("path").reset_index(drop=True)     # ترتيب ثابت

ok = df[~df["corrupted"]]
dup = ok.duplicated(subset="hash", keep="first")
n_labels = ok.groupby("hash")["label"].nunique()
conflict = n_labels[n_labels > 1].index

bad = set(df.loc[df["corrupted"], "path"])
bad |= set(ok.loc[dup, "path"])
bad |= set(ok.loc[ok["hash"].isin(conflict), "path"])

clean_df = df[~df["path"].isin(bad)].drop(columns="corrupted").reset_index(drop=True)
print(len(df), "|", len(bad), "|", len(clean_df))     # المتوقع 54305 | 21 | 54284
clean_df.to_csv(BASE + "clean_df.csv", index=False)

In [ ]:
#6. التقسيم (مرة واحدة فقط)
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    clean_df, test_size=0.30, stratify=clean_df["label"], random_state=42)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)
train_df, val_df, test_df = (x.reset_index(drop=True) for x in (train_df, val_df, test_df))

print("الأحجام:", len(train_df), len(val_df), len(test_df))   # ~37998 / 8143 / 8143
print("أقل عدد صور لفئة في test:", test_df["label"].value_counts().min())

train_df.to_csv(BASE + "train.csv", index=False)
val_df.to_csv(BASE + "val.csv", index=False)
test_df.to_csv(BASE + "test.csv", index=False)

In [ ]:
#7- فحص GPU

import tensorflow as tf
assert tf.config.list_physical_devices("GPU"), "فعّل GPU: Runtime ← Change runtime type"
print("GPU OK، TensorFlow", tf.__version__)      # سجّل رقم الإصدار

In [ ]:
#8. الإعدادات وترميز الفئات

import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

IMG = 128
BATCH = 64
AUTOTUNE = tf.data.AUTOTUNE

classes = sorted(train_df["label"].unique())
class2idx = {c: i for i, c in enumerate(classes)}
for part in (train_df, val_df, test_df):
    part["y"] = part["label"].map(class2idx)
NUM_CLASSES = len(classes)
print(NUM_CLASSES)      # 38

In [ ]:
#9. أوزان الفئات
from sklearn.utils.class_weight import compute_class_weight

w = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_df["y"].values)
class_weight = dict(enumerate(w))
print(round(w.min(), 2), round(w.max(), 2))     # تقريباً 0.26 إلى 9


In [ ]:
#10- pipeline الصور

def load(path, y):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG, IMG))
    return img, y

def make_ds(df, training):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, df["y"].values))
    if training:
        ds = ds.shuffle(len(df), seed=42)
    ds = ds.map(load, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_df, True)
val_ds   = make_ds(val_df, False)
test_ds  = make_ds(test_df, False)

x, y = next(iter(train_ds))
print(x.shape, y.shape)        # (64, 128, 128, 3) (64,)

In [ ]:
#11- بناء النموذج
# هنا استخدمت الـ augmentation  لأنه يساعد النموذج على التعامل مع اختلاف الإضاءة والموضع.

augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom((-0.3, 0.2)),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomBrightness(0.3),
    layers.RandomContrast(0.3),
], name="augment")

inputs = keras.Input((IMG, IMG, 3))
x = augment(inputs)
x = layers.Rescaling(1./255)(x)

for filters in (32, 64, 128, 256):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu", name="embedding")(x)
#بعد انتهاء التدريب نأخذ "بصمة" كل صورة من داخل النموذج (طبقة Dense(256))، ونقيس قربها من صور التدريب. الصورة القريبة تُقبل، والبعيدة تُرفض
#وهذا يسمح لنا باستخراج البصمة من داخل النموذج، ويبقى الاسم محفوظاً عند حفظ الملف وتحميله.

x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

cnn = keras.Model(inputs, outputs)
cnn.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])
cnn.summary()

In [ ]:
#12. التدريب (محمي من الانقطاع)


import shutil
CKPT = BASE + "cnn_final.keras"
LOG = BASE + "train_log_final.csv"
BACKUP = BASE + "backup_final"

RESUME = False       # اجعلها True فقط إذا انقطعت الجلسة وتريد المتابعة

if not RESUME:
    shutil.rmtree(BACKUP, ignore_errors=True)
    if os.path.exists(LOG): os.remove(LOG)

best_prev = float(pd.read_csv(LOG)["val_accuracy"].max()) if RESUME and os.path.exists(LOG) else None

cb = [
    keras.callbacks.ModelCheckpoint(CKPT, monitor="val_accuracy", save_best_only=True,
                                    initial_value_threshold=best_prev),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2),
    keras.callbacks.BackupAndRestore(BACKUP),
    keras.callbacks.CSVLogger(LOG, append=RESUME),
]

h_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=30,
                callbacks=cb, class_weight=class_weight, verbose=2)

In [ ]:
#13. تقييم validation

from sklearn.metrics import f1_score, classification_report

pred = np.argmax(cnn.predict(val_ds), axis=1)
y_val = val_df["y"].values
print("val accuracy :", (pred == y_val).mean())
print("val macro F1 :", f1_score(y_val, pred, average="macro"))
print(classification_report(y_val, pred, target_names=classes, digits=3))

In [ ]:
#الخلية 14: بصمات صور التدريب والـ validation

emb_model = keras.Model(cnn.input, cnn.get_layer("embedding").output)

def embeddings(df):
    f = emb_model.predict(make_ds(df, False), verbose=0)
    return f / (np.linalg.norm(f, axis=1, keepdims=True) + 1e-9)    # تطبيع L2

f_train = embeddings(train_df)
f_val = embeddings(val_df)

rng = np.random.RandomState(0)
ref = f_train[rng.choice(len(f_train), 10000, replace=False)].astype("float32")   # مرجع: 10000 صورة تدريب

def knn_score(f, ref, k=5, chunk=1000):
    out = []
    for s in range(0, len(f), chunk):
        sim = f[s:s+chunk] @ ref.T                               # تشابه cosine مع كل صور المرجع
        out.append(np.partition(sim, -k, axis=1)[:, -k:].mean(axis=1))   # متوسط أعلى 5 تشابهات
    return np.concatenate(out)

In [ ]:
#15 - ضبط العتبة وقياسها


import glob, pandas as pd

s_val = knn_score(f_val, ref)
ACCEPT = 0.95                                        # نقبل 95% من صور validation
thr_sim = float(np.percentile(s_val, (1 - ACCEPT) * 100))
print("العتبة:", round(thr_sim, 3))

def load_external(folder):
    files = [f for f in glob.glob(folder + "/**/*", recursive=True)
             if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]
    return pd.DataFrame({"path": files, "y": 0})

ext_df = load_external(BASE + "external")
s_ext = knn_score(embeddings(ext_df), ref)

print("عدد الصور الخارجية:", len(ext_df))
print("نسبة قبول صور validation :", (s_val >= thr_sim).mean())
print("نسبة رفض الصور الخارجية  :", (s_ext < thr_sim).mean())
print("درجات validation (1%, 5%, 50%):", np.percentile(s_val, [1, 5, 50]).round(3))
print("درجات الخارجية (50%, 95%, 99%):", np.percentile(s_ext, [50, 95, 99]).round(3))

In [ ]:
#16 - دالة التنبؤ مع الرفض



multi = keras.Model(cnn.input, [cnn.output, cnn.get_layer("embedding").output])

def predict_open(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG, IMG))                 # بدون قسمة على 255
    probs, emb = multi.predict(tf.expand_dims(img, 0), verbose=0)
    emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)
    score = float(knn_score(emb, ref)[0])
    if score < thr_sim:
        return {"label": "غير معروف", "score": round(score, 3)}
    i = int(np.argmax(probs[0]))
    return {"label": classes[i], "confidence": round(float(probs[0][i]), 3), "score": round(score, 3)}

print(predict_open(test_df["path"].iloc[0]), "| الصحيح:", test_df["label"].iloc[0])

In [ ]:
#17- ملفات إضافية للسيرفر

import json
np.save("/content/ref_embeddings.npy", ref)                    # ~10 MB
json.dump({"thr_sim": thr_sim, "k": 5}, open("/content/ood.json", "w"))

from google.colab import files
files.download("/content/ref_embeddings.npy")
files.download("/content/ood.json")